# 00.07 — Bayes' Theorem: Updating Beliefs with Evidence

**Module:** 00 — Math Foundations  
**Prerequisites:** 00.01–00.06  
**Difficulty:** introductory  
**Estimated time:** approximately 50–70 minutes

This lesson uses a fully synthetic, deterministic two-outcome example. It does not make medical, operational, causal, performance, or real-world claims.

By the end, you can distinguish inverse conditional probabilities, construct a joint table, compute a marginal denominator, derive Bayes' theorem, implement it with NumPy, verify it with exact fractions and counts, and explain how a prior changes a posterior.

## 1. The Problem

If an unusual detector fires, how should we combine what was likely before it ran with how the detector behaves to estimate the event after the evidence?

`P(E|H)` asks how often evidence appears when the hypothesis is present. `P(H|E)` asks how plausible the hypothesis is after observing the evidence. They condition in opposite directions, so they are not interchangeable.

## 2. Toy Example or Dataset

Let `H` mean the synthetic event is present and `E` mean the detector fires. The outcomes are exhaustive: `[H, not H]` and `[E, not E]`.

The invented prior is `P(H)=0.01`, `P(not H)=0.99`. The conditional table has rows `[H, not H]` and columns `[E, not E]`:

| | E | not E |
|---|---:|---:|
| H | 0.90 | 0.10 |
| not H | 0.05 | 0.95 |

A deterministic population of 20,000 gives counts `[[180, 20], [990, 18810]]`.

In [ ]:
import numpy as np

priors = np.array([0.01, 0.99], dtype=np.float64)
likelihood = np.array([[0.90, 0.10], [0.05, 0.95]], dtype=np.float64)
counts = np.array([[180, 20], [990, 18810]], dtype=np.int64)
priors, likelihood, counts

## 3. Intuition Before Math

Among 20,000 deterministic cases, 180 true firings and 990 false firings occur. Therefore the fraction of firings associated with `H` is `180/(180+990)=180/1170=2/13≈0.1538`.

The **prior** describes belief before evidence, the **likelihood** describes evidence conditional on a hypothesis, the **marginal** is the overall evidence probability, and the **posterior** is belief after evidence. The low prior matters: a 5% firing rate among the much larger `not H` group contributes many firings.

## 4. Mathematical Derivation

For `P(B)>0`, conditional probability is `P(A|B)=P(A and B)/P(B)`. Rearranging gives the product rule `P(A and B)=P(A|B)P(B)`.

The evidence marginal sums the two hypothesis paths:

`P(E)=P(E|H)P(H)+P(E|not H)P(not H)`.

Substitution into the definition gives Bayes' theorem:

`P(H|E)=P(E|H)P(H)/P(E)`.

No independence assumption is needed: the conditional table defines the detector behavior.

## 5. Manual Walkthrough

The four joint cells are:

- `P(H and E)=(0.90)(0.01)=0.009`
- `P(H and not E)=(0.10)(0.01)=0.001`
- `P(not H and E)=(0.05)(0.99)=0.0495`
- `P(not H and not E)=(0.95)(0.99)=0.9405`

The denominator is `P(E)=0.009+0.0495=0.0585=117/2000`, so `P(H|E)=0.009/0.0585=2/13≈0.153846`. Its complement is `11/13≈0.846154`. For silent evidence, `P(H|not E)=2/1883≈0.00106213` and the complement is `1881/1883`.

In [ ]:
joint = priors[:, None] * likelihood
marginal = joint.sum(axis=0)
if not np.all(marginal > 0):
    raise ValueError("Posterior denominators must be positive")
posterior = joint / marginal[None, :]

joint, marginal, posterior

The synthetic diagonal mass `0.009+0.9405=0.9495` is an overall accuracy-like quantity for this table, not `P(H|E)`. A posterior is not a causal claim, and the invented prior is not an empirical prevalence.

## 6. Teaching Implementation

NumPy keeps the axis order explicit: rows are hypotheses `[H, not H]`, columns are evidence `[E, not E]`. `joint = priors[:, None] * likelihood` broadcasts each prior across its conditional row. The marginal sums columns, then each posterior column is normalized by its positive denominator.

In [ ]:
print("shapes:", priors.shape, likelihood.shape, joint.shape, marginal.shape, posterior.shape)
print("P(H|E)=", posterior[0, 0], "  P(H|not E)=", posterior[0, 1])

## 7. Verification

Use `np.testing.assert_allclose` rather than floating-point equality. The frozen policy is `rtol=1e-12` and `atol=0.0`.

In [ ]:
RTOL, ATOL = 1e-12, 0.0
assert np.all(priors >= 0) and np.isclose(priors.sum(), 1.0)
assert np.all((likelihood >= 0) & (likelihood <= 1))
np.testing.assert_allclose(likelihood.sum(axis=1), [1.0, 1.0], rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(joint.sum(axis=1), priors, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(joint.sum(), 1.0, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(posterior.sum(axis=0), [1.0, 1.0], rtol=RTOL, atol=ATOL)
print("verification: PASS")

## 8. Independent Cross-Check

Exact arithmetic confirms the positive posterior without relying on the float64 pipeline. The deterministic count table gives the same answer: `180/1170=2/13`. Neither check uses random sampling or simulation.

In [ ]:
from fractions import Fraction

assert Fraction(9, 1000) / Fraction(117, 2000) == Fraction(2, 13)
assert Fraction(int(counts[0, 0]), int(counts[:, 0].sum())) == Fraction(2, 13)
print("exact fraction and integer-count checks: PASS")

## 9. Visual Explanation

![Synthetic probability tree and belief update](assets/bayes_tree_belief_update.png)

The left panel shows prior and likelihood branches becoming joint paths, then sums the positive-evidence paths to form the denominator. The right panel compares prior belief with posterior belief after observing `E`. Synthetic instructional probabilities only; colours are supplemented by labels, line styles, markers, and hatching.

## 10. Failure Modes

- **Inverse conditioning:** `P(E|H)=0.90` is not `P(H|E)=2/13`.
- **Ignoring the prior:** the rare prior and the 990 false firings matter.
- **Skipping the denominator:** `0.009` is a joint probability, not a posterior.
- **Confusing marginal and likelihood:** `P(E)=0.0585`, not `P(E|H)=0.90`.
- **Calling posterior accuracy:** `0.9495` and `0.1538` answer different questions.
- **Treating belief as certainty or causality:** `P(H|E)<1`, and updating does not establish cause.

The table is deterministic and synthetic; Bayes' theorem does not require simulation.

## 11. Exercises

1. Reconstruct the frozen joint table and verify its sum.
2. Compute `P(E)` and `P(H|E)` from the table.
3. Explain why `180/1170` agrees with `2/13`.
4. Repeat the calculation with `P(H)=0.20`, `P(E|H)=0.75`, and `P(E|not H)=0.10`; targets are `P(E)=0.23`, `P(H|E)=15/23`, and `P(not H|E)=8/23`.
5. Correct this statement: “The detector fires 90% of the time when H is present, so a firing means H is 90% likely.”

## 12. References

- OpenStax, *Introductory Statistics 2e*, conditional probability and product rule: https://openstax.org/books/introductory-statistics-2e/pages/3-3-two-basic-rules-of-probability
- OpenStax, *Principles of Data Science*, Bayes' theorem: https://openstax.org/books/principles-data-science/pages/3-4-bayes-theorem
- NumPy documentation, `testing.assert_allclose`: https://numpy.org/doc/stable/reference/generated/numpy.testing.assert_allclose.html
- MIT OpenCourseWare, Introduction to Probability: https://ocw.mit.edu/courses/18-05-introduction-to-probability-spring-2014/

Continuous Bayes, parameter estimation, Naive Bayes, odds, sequential updating, causality, evaluation metrics, and real-world claims are outside this lesson.